## Assignment  - The Transformer Decoder

###  **Objective**

Implement the two key remaining components of the "Attention Is All You Need" Transformer:

1.  **Positional Encoding (`PositionalEncoding`):** A non-trainable layer to inject sequence order information.
2.  **Decoder Layer (`DecoderLayer`):** The full decoder block, which includes masked self-attention, cross-attention, and a feed-forward network.

###  **Assignment Structure**

This assignment will contain:

1.  **Setup & Prerequisite Code:** Imports, configuration variables, and the *completed* solutions for `scaled_dot_product_attention` and `MultiHeadAttention` from Week 1.
2.  **Part 1: Positional Encoding:** A skeleton class for the student to complete, followed by verification code and questions.
3.  **Part 2: Decoder Layer:** A skeleton class for the student to complete, followed by verification code and questions.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- Configuration for testing ---
# Use these global variables in your verification steps
BATCH_SIZE = 8
SEQ_LENGTH = 12
D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048  # Dimension for the Feed-Forward Network
DROPOUT = 0.1
MAX_LEN = 5000 # Default max sequence length for PE

# --- Reproducibility ---
# Set the seed for all random operations
torch.manual_seed(42)

print("="*50)
print("TRANSFORMER DECODER")
print(f"PyTorch Version: {torch.__version__}")
print("="*50)

# --- Prerequisite Code (Solutions from Week 1) ---
# (Provided to the student, no work required here)
print("Loading prerequisite code from assignment 1...")

def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Calculates the Scaled Dot-Product Attention.

    Args:
        query (torch.Tensor): Shape (batch_size, num_heads, seq_len_q, head_dim)
        key (torch.Tensor): Shape (batch_size, num_heads, seq_len_k, head_dim)
        value (torch.Tensor): Shape (batch_size, num_heads, seq_len_v, head_dim)
        mask (torch.Tensor, optional): Shape (batch_size, 1, 1, seq_len_k) or
                                       (batch_size, 1, seq_len_q, seq_len_k)

    Returns:
        torch.Tensor: Output tensor, shape (batch_size, num_heads, seq_len_q, head_dim)
        torch.Tensor: Attention weights, shape (batch_size, num_heads, seq_len_q, seq_len_k)
    """
    # 1. Get the dimension of the key (d_k)
    d_k = key.size(-1)

    # 2. Calculate scores (Q * K^T)
    # Expected shape: (batch_size, num_heads, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(2, 3))

    # 3. Scale the scores
    scaled_scores = scores / (d_k ** 0.5)

    # 4. Apply mask (if provided)
    if mask is not None:
        # Fill with a very small number (-1e9) where mask is 0
        scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)

    # 5. Apply softmax to get attention weights
    # (Apply along the last dimension)
    attention_weights = F.softmax(scaled_scores, -1)

    # 6. Get the final output (Attention_Weights * V)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # 1. Define the four linear layers
        # (wq, wk, wv for projections, wo for output)
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1. Pass Q, K, V through linear layers
        # Shape: (batch_size, seq_len, d_model)
        Q = self.wq(query)
        K = self.wk(key)
        V = self.wv(value)

        # 2. Reshape Q, K, V for multi-head processing
        # Change shape from (batch_size, seq_len, d_model) to
        # (batch_size, num_heads, seq_len, head_dim)
        # Hint: Use .view() and .transpose(1, 2)
        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Call scaled_dot_product_attention
        # (This is the function you wrote in Part 1)
        attention_output, _ = scaled_dot_product_attention(Q, K, V, mask)

        # 4. Concatenate heads back
        # Reshape from (batch_size, num_heads, seq_len, head_dim) to
        # (batch_size, seq_len, d_model)
        # Hint: Use .transpose(1, 2), .contiguous(), and .view()
        attention_output = attention_output.transpose(1, 2).contiguous()
        concatenated_output = attention_output.view(batch_size, -1, self.d_model)

        # 5. Pass through final linear layer (wo)
        final_output = self.wo(concatenated_output)

        return final_output

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        # 1. Define the two linear layers and activation
        # Layer 1: d_model -> d_ff
        # Layer 2: d_ff -> d_model
        # Activation: ReLU
        self.fc1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout()
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # Pass x through fc1, relu, dropout, fc2
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("Prerequisite code loaded successfully.")

TRANSFORMER DECODER
PyTorch Version: 2.10.0+cpu
Loading prerequisite code from assignment 1...
Prerequisite code loaded successfully.


-----

### **Part 1: Positional Encoding**

In [ ]:
print("\n--- Part 1: Positional Encoding ---")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        """
        Initializes the PositionalEncoding module.

        Args:
            d_model (int): The embedding dimension.
            max_len (int): The maximum sequence length.
        """
        super(PositionalEncoding, self).__init__()

        # 1. Create a matrix 'pe' of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # 2. Create 'position' tensor (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 3. Create 'div_term' for the denominator
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # 4. Calculate sin for even indices
        pe[:, 0::2] = torch.sin(position * div_term)

        # 5. Calculate cos for odd indices
        pe[:, 1::2] = torch.cos(position * div_term)

        # 6. Add a batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)

        # 7. Register 'pe' as a buffer. This makes it part of the model's
        # state_dict, but not a trainable parameter.
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Adds positional encoding to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, d_model)

        Returns:
            torch.Tensor: Output tensor with added positional encodings
        """
        # 1. Add the positional encodings to the input x
        # We only need the PEs up to x's sequence length
        x =  x + self.pe[:, :x.size(1)]

        # 2. Return the result
        return x

# --- Verification for Part 1 ---
print("--- Running Verification for Part 1 ---")
try:
    torch.manual_seed(42)
    pe_layer = PositionalEncoding(D_MODEL, max_len=50)
    x_test = torch.zeros(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    output = pe_layer(x_test)

    assert output.shape == x_test.shape
    print(" (1a) Output shape is correct.")
    assert not torch.all(output == 0)
    print(" (1b) Positional encodings were successfully added.")
    assert torch.allclose(output[0], output[1])
    print(" (1c) PEs are consistent across the batch.")
    assert not torch.allclose(output[0, 0, :], output[0, 1, :])
    print(" (1d) PEs differ by position.")
    assert len(list(pe_layer.parameters())) == 0 and 'pe' in pe_layer._buffers
    print(" (1e) PE matrix is correctly registered as a buffer.")
    print(" Part 1 seems correct!")
except Exception as e:
    print(f" Part 1 Failed: {e}")


--- Part 1: Positional Encoding ---
--- Running Verification for Part 1 ---
 (1a) Output shape is correct.
 (1b) Positional encodings were successfully added.
 (1c) PEs are consistent across the batch.
 (1d) PEs differ by position.
 (1e) PE matrix is correctly registered as a buffer.
 Part 1 seems correct!


#### **Questions for Part 1**

**Q1: Conceptual Check**
Why is the positional encoding matrix `pe` registered as a **buffer** (using `self.register_buffer`) instead of a **parameter** (using `nn.Parameter`)?

  * (A) To ensure it is saved to the `state_dict` but **not** updated by the optimizer.
  * (B) To make the matrix multiplication in the `forward` pass run faster.
  * (C) To ensure it is automatically moved to the GPU along with the model.
  * (D) To allow it to be fine-tuned during training.



**Q2: Conceptual Check**
Why do we **add** the positional encodings to the word embeddings instead of, for example, concatenating them?

  * (A) Adding is computationally cheaper than concatenating.
  * (B) Concatenating would double the `d_model` dimension, which is inefficient.
  * (C) Adding allows the model to easily learn relative positions, as `PE(pos+k)` can be represented as a linear function of `PE(pos)`.
  * (D) Both (B) and (C) are key reasons.


**Q3: Value Check**
Run the code block below. What is the value of the element at `pe_val[0, 5, 10]`? (This tests the encoding for the 6th token, at the 10th dimension index).

In [ ]:
# Code for Q3
torch.manual_seed(42)
pe_layer_q3 = PositionalEncoding(d_model=512, max_len=100)
pe_val = pe_layer_q3.pe
# Get the value at position 5, dimension 10
print(f"Q3 Value: {pe_val[0, 5, 10].item()}")

Q3 Value: -0.8599746227264404



-----

### **Part 2: The Complete Decoder Layer**

In [ ]:
print("\n--- Part 2: The Complete Decoder Layer ---")

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=2048, dropout=0.1):
        super(DecoderLayer, self).__init__()

        # 1. Masked Self-Attention (same as Encoder's MHA)
        self.self_attn = MultiHeadAttention(d_model, num_heads)

        # 2. Cross-Attention (new layer for the decoder)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)

        # 3. Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

        # 4. LayerNorms (one for each sub-layer)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        # 5. Dropouts (one for each sub-layer)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, look_ahead_mask, padding_mask=None):
        """
        Passes inputs through the DecoderLayer.

        Args:
            x (torch.Tensor): Target sequence (from decoder input).
            encoder_output (torch.Tensor): Output from the encoder stack.
            look_ahead_mask (torch.Tensor): Causal mask for self-attention.
            padding_mask (torch.Tensor, optional): Mask for encoder padding.
        """

        # 1. Masked Self-Attention (Add & Norm)
        # Q, K, V all come from 'x' (the decoder's input)
        # Use the 'look_ahead_mask' here
        attn_output = self.self_attn(
            query=x, key=x, value=x,
            mask=look_ahead_mask
        )
        x = self.norm1(x + self.dropout1(attn_output))

        # 2. Cross-Attention (Add & Norm)
        # Query comes from 'x' (output of previous layer)
        # Key & Value come from 'encoder_output'
        # Use the 'padding_mask' here (if provided)
        cross_attn_output = self.cross_attn(
            query=x,
            key=attn_output,
            value=attn_output,
            mask=padding_mask
        )
        x = self.norm2(x + self.dropout2(cross_attn_output))

        # 3. Feed Forward (Add & Norm)
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))

        return x

# --- Verification for Part 2 ---
print("--- Running Verification for Part 2 ---")
def create_look_ahead_mask(size):
    # Creates a (1, 1, size, size) mask
    mask = (torch.triu(torch.ones(size, size), diagonal=1) == 0)
    return mask.unsqueeze(0).unsqueeze(1)

try:
    torch.manual_seed(42)
    # Create test inputs
    decoder_layer = DecoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
    x_test = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    enc_out_test = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL) # Encoder output
    look_ahead_mask_test = create_look_ahead_mask(SEQ_LENGTH)

    # Run forward pass
    output = decoder_layer(x_test, enc_out_test, look_ahead_mask_test)

    assert output.shape == x_test.shape
    print(" (2a) Final output shape is correct.")

    # Test that the look-ahead mask has an effect
    output_no_mask = decoder_layer(x_test, enc_out_test, look_ahead_mask=None)
    assert not torch.allclose(output, output_no_mask, atol=1e-5)
    print("(2b) Look-ahead mask is being applied.")

    # Test that encoder output has an effect
    enc_out_test_zeros = torch.zeros_like(enc_out_test)
    output_zero_enc = decoder_layer(x_test, enc_out_test_zeros, look_ahead_mask_test)
    assert not torch.allclose(output, output_zero_enc, atol=1e-5)
    print(" (2c) Decoder output depends on encoder output.")

    print(" Part 2 seems correct!")
except Exception as e:
    print(f" Part 2 Failed: {e}")


--- Part 2: The Complete Decoder Layer ---
--- Running Verification for Part 2 ---
 (2a) Final output shape is correct.
(2b) Look-ahead mask is being applied.
 (2c) Decoder output depends on encoder output.
 Part 2 seems correct!


#### **Questions for Part 2**

**Q4: Conceptual Check**
In the **Cross-Attention** sub-layer (`self.cross_attn`), where do the Query (Q), Key (K), and Value (V) inputs come from?

  * (A) Q, K, and V all come from the `encoder_output`.
  * (B) Q, K, and V all come from the decoder's input `x`.
  * (C) Q comes from `x`; K and V come from the `encoder_output`.
  * (D) Q and K come from `x`; V comes from the `encoder_output`.



**Q5: Conceptual Check**
What is the purpose of the `look_ahead_mask` that is passed to the **Masked Self-Attention** (`self.self_attn`) layer?

  * (A) It prevents the model from attending to padding tokens in the *encoder* output.
  * (B) It prevents the model from "cheating" by attending to future tokens in the *decoder's* own target sequence.
  * (C) It randomly masks out tokens to improve generalization, like dropout.
  * (D) It forces the decoder to pay more attention to the cross-attention output.



**Q6: Conceptual Check**
How many `MultiHeadAttention` modules and `LayerNorm` modules are there in a single `DecoderLayer`?

  * (A) 1 MHA module and 2 LayerNorm modules.
  * (B) 2 MHA modules and 2 LayerNorm modules.
  * (C) 2 MHA modules and 3 LayerNorm modules.
  * (D) 3 MHA modules and 3 LayerNorm modules.


**Q7: Total Parameters**
Run the code block below to calculate the total number of **trainable parameters** in one `DecoderLayer` (using our global config: `D_MODEL=512`, `NUM_HEADS=8`, `D_FF=2048`).

In [ ]:
# Code for Q7
torch.manual_seed(42)
decoder_layer_q7 = DecoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
total_params = sum(p.numel() for p in decoder_layer_q7.parameters() if p.requires_grad)
print(f"Q7 Total Parameters: {total_params}")

Q7 Total Parameters: 4204032




**Q8: Final Value Check**
Using the `decoder_layer` and inputs initialized in the verification block (with `seed=42`), what is the **sum** of the final `output` tensor?

In [ ]:
# Code for Q8
# (This uses the exact same variables from the Verification block)
try:
    torch.manual_seed(42)
    decoder_layer_q8 = DecoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
    x_test_q8 = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    enc_out_test_q8 = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
    look_ahead_mask_test_q8 = create_look_ahead_mask(SEQ_LENGTH)

    output_q8 = decoder_layer_q8(x_test_q8, enc_out_test_q8, look_ahead_mask_test_q8)
    print(f"Q8 Final Output Sum: {output_q8.sum().item()}")
except Exception as e:
    print(f"Q8 Failed to run: {e}")

Q8 Final Output Sum: 0.0
